# 05 · Descriptivos del subcorpus de salud
**Proyecto:** Salud en la esfera pública — análisis de tweets en español  
**Equivalente original:** `6_descriptivos_ciencia.ipynb` + `6_1_nubes_palabras_comparativas.ipynb`

---

### Diferencias con el paper

| Aspecto | Paper | Este notebook |
|---|---|---|
| Unidad | Fragmentos (chunks) de columnas | Tweets completos |
| Metadatos | Autor, Diario | Author_Normalized, Entidad |
| Conteo fragmentos/col | Chunks de ciencia por columna | Tweets de salud por autor |
| Proporción | Ciencia/total chunks por columna | Salud/total tweets por autor |
| Subcategorías | Prefijo `Ciencia_` | Prefijo `Salud_` |
| Tabla reducida | LaTeX (.tex) | Excel (.xlsx) |

**Entradas:**
- `resultadosPropios/salud_tweets_final.parquet`
- `resultadosPropios/corpus_cleaned.parquet`

**Salidas (figuras):**
- `fig_dist_tweets_salud_por_autor.png`
- `fig_porcentaje_mensual_salud_barras.png`
- `fig_porcentaje_mensual_salud_linea.png`
- `fig_autores_salud_simple.png`
- `fig_autores_salud_comparativo.png`
- `fig_subcategorias_tweets.png`
- `fig_autores_por_subcategoria.png`
- `fig_entidad_salud.png`
- `nube_corpus_general.png`
- `nube_subcorpus_salud.png`

**Salidas (tablas):**
- `subcategorias_tweets.csv`
- `tabla_reducida_frecuencias.xlsx`
- `frecuencias_corpus_lematizado.parquet`
- `frecuencias_salud_lematizado.parquet`
- `tabla_frecuencias_salud.xlsx` (multi-hoja resumen final)


## 0 · Configuración

In [ ]:
# ============================================================
# CELL 0 — CONFIG
# ============================================================
from pathlib import Path

DATA_PROCESSED = Path(r'C:\Users\afpue\Documents\GitHub\icare\kMetodo\resultadosPropios')

COL_AUTOR   = 'Author_Normalized'
COL_ENTIDAD = 'Entidad'
COL_FECHA   = 'Fecha'

print('[CONFIG] OK')
print(f'  DATA_PROCESSED : {DATA_PROCESSED.resolve()}')


## 1 · Imports y carga de datos

In [ ]:
# ============================================================
# CELL 1 — IMPORTS Y CARGA
# Equivalente a: ciencia_chunks.xlsx + corpus_cleaned.xlsx + chunks.parquet
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
import spacy, re, subprocess, sys
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('crest')

# Subcorpus de salud + corpus completo
tweets_salud = pd.read_parquet(DATA_PROCESSED / 'salud_tweets_final.parquet')
corpus       = pd.read_parquet(DATA_PROCESSED / 'corpus_cleaned.parquet')

corpus['Fecha'] = pd.to_datetime(corpus['Fecha'], errors='coerce')

# Merge metadatos al subcorpus de salud
cols_meta = ['id_doc', 'Author_Normalized', 'Entidad', 'Fecha',
             'Texto_limpio', 'Sentimiento', 'Polaridad']
cols_meta = [c for c in cols_meta if c in corpus.columns]

tweets_salud = tweets_salud.merge(corpus[cols_meta], on='id_doc', how='left')
tweets_salud['Fecha'] = pd.to_datetime(tweets_salud['Fecha'], errors='coerce')
tweets_salud['Anio'] = tweets_salud['Fecha'].dt.year
tweets_salud['Mes']  = tweets_salud['Fecha'].dt.month

print(f'Corpus completo    : {len(corpus):,} tweets')
print(f'Subcorpus de salud : {len(tweets_salud):,} tweets '
      f'({len(tweets_salud)/len(corpus)*100:.1f}%)')
tweets_salud.head(3)


## 2 · Estadísticos básicos
Equivalente a las celdas 10–13 del paper.

In [ ]:
# ============================================================
# CELL 2 — ESTADISTICOS BASICOS
# Equivalente a celdas 10-13 del paper
# ============================================================

n_total      = len(corpus)
n_salud      = len(tweets_salud)
n_docs_total = corpus['id_doc'].nunique()
n_docs_salud = tweets_salud['id_doc'].nunique()

print('=' * 55)
print('ESTADISTICOS DEL SUBCORPUS DE SALUD')
print('=' * 55)
print(f'  Total tweets en corpus         : {n_total:,}')
print(f'  Tweets con mencion de salud    : {n_salud:,}')
# Equivalente a: (7267/62651)*100
print(f'  % tweets de salud / total      : {n_salud/n_total*100:.2f}%')
print(f'  Autores distintos (salud)      : {tweets_salud[COL_AUTOR].nunique():,}')
print(f'  Autores distintos (corpus)     : {corpus[COL_AUTOR].nunique():,}')
# Equivalente a: (4154/13676)*100
print(f'  % autores con salud / total    : '
      f'{tweets_salud[COL_AUTOR].nunique()/corpus[COL_AUTOR].nunique()*100:.2f}%')
print(f'  Rango de fechas                : '
      f'{tweets_salud[COL_FECHA].min().date()} -> {tweets_salud[COL_FECHA].max().date()}')
print('=' * 55)


## 3 · Distribución de tweets de salud por autor
Equivalente a las celdas 14–18 del paper:  
cuántos fragmentos de ciencia tiene cada columna → cuántos tweets de salud tiene cada autor.

In [ ]:
# ============================================================
# CELL 3 — TWEETS DE SALUD POR AUTOR
# Equivalente a celdas 14-15 del paper
# ============================================================

# Cuantos tweets de salud tiene cada autor (equiv. cuántos chunks de ciencia por columna)
conteo_por_autor = tweets_salud[COL_AUTOR].value_counts().reset_index()
conteo_por_autor.columns = [COL_AUTOR, 'num_tweets_salud']

# Distribucion: cuantos autores tienen 1, 2, 3... tweets de salud (equiv. cell 15)
distribucion = tweets_salud[COL_AUTOR].value_counts().value_counts().sort_index()
porcentajes_dist = distribucion / distribucion.sum() * 100

# Solo mostrar hasta el percentil 95 para que el grafico sea legible
p95 = int(np.percentile(conteo_por_autor['num_tweets_salud'], 95))
dist_plot = distribucion[distribucion.index <= p95]
pct_plot  = porcentajes_dist[porcentajes_dist.index <= p95]

plt.figure(figsize=(10, 5))
ax = sns.barplot(x=dist_plot.index, y=dist_plot.values, color='teal')
for i, (x, y) in enumerate(zip(dist_plot.index, dist_plot.values)):
    ax.text(i, y + 0.5, f'{pct_plot.iloc[i]:.1f}%', ha='center', va='bottom', fontsize=8)
plt.title('Distribucion de autores segun numero de tweets de salud (P95)')
plt.xlabel('Numero de tweets de salud por autor')
plt.ylabel('Numero de autores')
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'fig_dist_tweets_salud_por_autor.png', dpi=300)
plt.show()

print(distribucion.head(10).to_frame('num_autores').assign(porcentaje=porcentajes_dist.head(10).round(2)))


In [ ]:
# ============================================================
# CELL 4 — PROPORCION SALUD / TOTAL POR AUTOR
# Equivalente a celdas 16-18 del paper (proporcion ciencia/chunks por columna)
# ============================================================

tweets_por_autor_total = corpus[COL_AUTOR].value_counts().reset_index()
tweets_por_autor_total.columns = [COL_AUTOR, 'num_tweets_total']

df_ratio = conteo_por_autor.merge(tweets_por_autor_total, on=COL_AUTOR, how='left')
df_ratio['proporcion_salud'] = df_ratio['num_tweets_salud'] / df_ratio['num_tweets_total']

print('Estadisticos de proporcion de salud por autor:')
print(df_ratio['proporcion_salud'].describe().round(4))

promedio_tweets_total  = df_ratio['num_tweets_total'].mean()
promedio_tweets_salud  = df_ratio['num_tweets_salud'].mean()
promedio_prop_salud    = df_ratio['proporcion_salud'].mean()

print(f'\nPromedio de tweets totales por autor  : {promedio_tweets_total:.2f}')
print(f'Promedio de tweets de salud por autor : {promedio_tweets_salud:.2f}')
print(f'Promedio de proporcion de salud       : {promedio_prop_salud*100:.2f}%')


## 4 · Distribución temporal
Equivalente a las celdas 20–21 del paper.

In [ ]:
# ============================================================
# CELL 5 — DISTRIBUCION TEMPORAL
# Equivalente a celdas 20-21 del paper
# ============================================================

corpus['AnoMes']       = corpus[COL_FECHA].dt.to_period('M')
tweets_salud['AnoMes'] = tweets_salud[COL_FECHA].dt.to_period('M')

totales   = corpus.groupby('AnoMes')['id_doc'].nunique()
con_salud = tweets_salud.groupby('AnoMes')['id_doc'].nunique()

mensual = pd.DataFrame({'total_tweets': totales, 'tweets_salud': con_salud}).fillna(0)
mensual['porcentaje'] = (mensual['tweets_salud'] / mensual['total_tweets'] * 100).round(2)
mensual.index = mensual.index.to_timestamp()

# Grafica 1: barras apiladas
plt.figure(figsize=(14, 6))
plt.bar(mensual.index, mensual['total_tweets'],
        color='lightsteelblue', label='Total tweets', width=25)
plt.bar(mensual.index, mensual['tweets_salud'],
        color='darkorange', label='Con mencion de salud', width=25)
for x, y_s, pct in zip(mensual.index, mensual['tweets_salud'], mensual['porcentaje']):
    if y_s > 0:
        plt.text(x, y_s + 2, f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)
plt.title('Monthly percentage of tweets mentioning health')
plt.ylabel('Number of tweets')
plt.xlabel('Month')
plt.legend()
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'fig_porcentaje_mensual_salud_barras.png', dpi=300)
plt.show()

# Grafica 2: linea de porcentaje
plt.figure(figsize=(14, 6))
plt.plot(mensual.index, mensual['porcentaje'], marker='o', linestyle='-', color='darkorange', linewidth=2)
for x, y in zip(mensual.index, mensual['porcentaje']):
    if not np.isnan(y):
        plt.text(x, y + 0.5, f'{y:.1f}%', ha='center', va='bottom', fontsize=8)
plt.title('Monthly percentage of tweets mentioning health')
plt.ylabel('% of tweets')
plt.xlabel('Month')
plt.grid(alpha=0.3)
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'fig_porcentaje_mensual_salud_linea.png', dpi=300)
plt.show()
mensual.tail(10)


## 5 · Autores con más tweets de salud
Equivalente a las celdas 22–25 del paper.

In [ ]:
# ============================================================
# CELL 6 — TOP AUTORES (gráfico simple, equiv. cell 23)
# ============================================================

autores_salud = (tweets_salud.groupby(COL_AUTOR)['id_doc'].nunique()
    .sort_values(ascending=False).reset_index(name='n_tweets_salud'))
autores_total = (corpus.groupby(COL_AUTOR)['id_doc'].nunique()
    .reset_index(name='total_columnas'))
autores_merge = autores_salud.merge(autores_total, on=COL_AUTOR, how='left')
autores_merge['porcentaje_salud'] = (
    autores_merge['n_tweets_salud'] / autores_merge['total_columnas'] * 100).round(2)

top_autores = autores_merge.head(15)

plt.figure(figsize=(9, 6))
orden = top_autores.sort_values('n_tweets_salud', ascending=False)[COL_AUTOR]
sns.barplot(data=top_autores, y=COL_AUTOR, x='n_tweets_salud', order=orden, color='lightcoral')
for i, (n, p) in enumerate(zip(top_autores['n_tweets_salud'], top_autores['porcentaje_salud'])):
    plt.text(n + 0.2, i, f'{p:.1f}%', va='center', fontsize=9)
plt.title('Autores con mas tweets que mencionan salud')
plt.xlabel('Numero de tweets con menciones de salud')
plt.ylabel('Autor')
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'fig_autores_salud_simple.png', dpi=300)
plt.show()


In [ ]:
# ============================================================
# CELL 7 — TOP AUTORES (gráfico comparativo total vs salud, equiv. cell 24)
# ============================================================

top_sorted = top_autores.sort_values('n_tweets_salud', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(top_sorted[COL_AUTOR], top_sorted['total_columnas'],
         color='mistyrose', label='Total tweets')
plt.barh(top_sorted[COL_AUTOR], top_sorted['n_tweets_salud'],
         color='lightcoral', label='Con menciones de salud')
for i, (n, t, p) in enumerate(zip(
    top_sorted['n_tweets_salud'], top_sorted['total_columnas'], top_sorted['porcentaje_salud']
)):
    plt.text(t + 1, i, f'{p:.1f}%', va='center', fontsize=8)
plt.title('Autores con mas tweets que mencionan salud')
plt.xlabel('Numero de tweets')
plt.legend()
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'fig_autores_salud_comparativo.png', dpi=300)
plt.show()
top_autores


## 6 · Por subcategorías
Equivalente a las celdas 26–31 del paper.

In [ ]:
# ============================================================
# CELL 8 — SUBCATEGORIAS: barplot + tabla + CSV (equiv. cells 28-30)
# ============================================================

subcats_cols = [c for c in tweets_salud.columns if c.startswith('Salud_')]
tweets_salud['subcategoria_max'] = tweets_salud[subcats_cols].idxmax(axis=1)

conteos     = tweets_salud['subcategoria_max'].value_counts()
total_s     = conteos.sum()
porcentajes = (conteos / total_s * 100).round(2)

tabla_subcats = pd.DataFrame({
    'Subcategoria': conteos.index,
    'N_tweets'    : conteos.values,
    'Porcentaje'  : porcentajes.values,
}).sort_values('N_tweets', ascending=False)

print('=' * 60)
print('TABLA DE SUBCATEGORIAS DE SALUD')
print('=' * 60)
print(f'Total tweets de salud: {total_s:,}')
print('-' * 60)
print(tabla_subcats.to_string(index=False))

# Grafico
conteos_ord = conteos.sort_values(ascending=True)
plt.figure(figsize=(12, 8))
ax = conteos_ord.plot.barh(color='indigo', width=0.9)
for i, (v, pct) in enumerate(zip(
    conteos_ord.values, porcentajes[conteos_ord.index].values
)):
    ax.text(v + v*0.01, i, f'{v:,} ({pct}%)', va='center', fontsize=9, fontweight='bold')
plt.title('Cantidad de tweets por subcategoria de salud', fontsize=14, fontweight='bold')
plt.xlabel('Numero de tweets')
plt.ylabel('Subcategoria')
plt.grid(axis='x', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'fig_subcategorias_tweets.png', dpi=300)
plt.show()

# Guardar CSV (equiv. Karen: subcategorias_cientificas.csv)
tabla_subcats.to_csv(DATA_PROCESSED / 'subcategorias_tweets.csv', index=False, encoding='utf-8-sig')
print('[GUARDADO] subcategorias_tweets.csv')


In [ ]:
# ============================================================
# CELL 9 — AUTORES POR SUBCATEGORIA (equiv. cell 31)
# ============================================================

autores_por_subcat = (
    tweets_salud.groupby('subcategoria_max')[COL_AUTOR]
    .nunique().sort_values(ascending=False)
)

print('Autores distintos por subcategoria:')
print(autores_por_subcat.to_string())

plt.figure(figsize=(10, 6))
ax = autores_por_subcat.sort_values().plot.barh(color='teal')
for i, v in enumerate(autores_por_subcat.sort_values().values):
    ax.text(v + 2, i, str(v), va='center', fontsize=9)
plt.title('Numero de autores distintos que mencionan cada subcategoria de salud')
plt.xlabel('Numero de autores (unicos)')
plt.ylabel('Subcategoria')
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'fig_autores_por_subcategoria.png', dpi=300)
plt.show()


## 7 · Distribución por tipo de cuenta
Equivalente adicional — reemplaza distribución por diario.

In [ ]:
# ============================================================
# CELL 10 — POR TIPO DE CUENTA (Entidad)
# ============================================================

entidad_salud = tweets_salud.groupby(COL_ENTIDAD)['id_doc'].nunique().reset_index(name='n_salud')
entidad_total = corpus.groupby(COL_ENTIDAD)['id_doc'].nunique().reset_index(name='n_total')
entidad_merge = entidad_salud.merge(entidad_total, on=COL_ENTIDAD, how='left')
entidad_merge['pct_salud'] = (entidad_merge['n_salud'] / entidad_merge['n_total'] * 100).round(1)
entidad_merge = entidad_merge.sort_values('n_salud', ascending=False)

top_ent = entidad_merge.sort_values('n_salud', ascending=True)
plt.figure(figsize=(10, 6))
plt.barh(top_ent[COL_ENTIDAD], top_ent['n_total'], color='lightsteelblue', label='Total tweets')
plt.barh(top_ent[COL_ENTIDAD], top_ent['n_salud'], color='steelblue', label='Con mencion de salud')
for i, (n, t, p) in enumerate(zip(top_ent['n_salud'], top_ent['n_total'], top_ent['pct_salud'])):
    plt.text(t + 1, i, f'{p:.1f}%', va='center', fontsize=8)
plt.title('Tweets de salud por tipo de cuenta')
plt.xlabel('Numero de tweets')
plt.legend()
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'fig_entidad_salud.png', dpi=300)
plt.show()
entidad_merge


## 8 · Nubes de palabras
Equivalente a las celdas 32–34 del paper + `6_1_nubes_palabras_comparativas.ipynb`.

In [ ]:
# ============================================================
# CELL 11 — NUBES DE PALABRAS
# Equivalente a celdas 33-34 del paper
# ============================================================

subprocess.run([sys.executable, '-m', 'spacy', 'download', 'es_core_news_sm'], capture_output=True)
nlp = spacy.load('es_core_news_sm')
STOP_WORDS = nlp.Defaults.stop_words

WC_PARAMS = dict(
    width=1200, height=700, background_color='white',
    stopwords=STOP_WORDS, max_words=150,
    collocations=False, prefer_horizontal=1.0,
    relative_scaling=0.3, max_font_size=120, min_font_size=8,
    random_state=42,
)

# Nube 1: subcorpus de salud (equiv. nube de fragmentos de ciencia)
texto_salud = ' '.join(tweets_salud['Texto_limpio'].dropna())
wc_salud = WordCloud(**WC_PARAMS, colormap='plasma').generate(texto_salud)
plt.figure(figsize=(10, 6))
plt.imshow(wc_salud, interpolation='bilinear')
plt.axis('off')
plt.title('Nube de palabras — Subcorpus de salud', fontsize=14)
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'nube_subcorpus_salud.png', dpi=300)
plt.show()

# Nube 2: corpus general (adicional comparativo)
texto_general = ' '.join(corpus['Texto_limpio'].dropna())
wc_general = WordCloud(**WC_PARAMS, colormap='winter').generate(texto_general)
plt.figure(figsize=(10, 6))
plt.imshow(wc_general, interpolation='bilinear')
plt.axis('off')
plt.title('Nube de palabras — Corpus general', fontsize=14)
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'nube_corpus_general.png', dpi=300)
plt.show()
print('[GUARDADO] nube_subcorpus_salud.png  +  nube_corpus_general.png')


## 9 · Tablas de palabras más frecuentes
Equivalente a las celdas 35–37 del paper.

In [ ]:
# ============================================================
# CELL 12 — FUNCION obtener_frecuencias
# Replica exacta de utils.obtener_frecuencias() del paper
# ============================================================

from tqdm import tqdm

def obtener_frecuencias(textos, stopwords, n=None, lematizar=False):
    """Replica de utils.obtener_frecuencias() del paper."""
    texto_unido = ' '.join(map(str, textos)).lower()
    tokens = re.findall(r'\b[a-záéíóúüñ]+\b', texto_unido)  # mismo regex que Karen
    tokens = [t for t in tokens if t not in stopwords and len(t) > 2]

    if not lematizar:
        return Counter(tokens).most_common(n)

    print('Lematizando textos...')
    lemas = []
    for doc in tqdm(nlp.pipe(tokens, batch_size=500, disable=['ner','parser'], n_process=2),
                    total=len(tokens), desc='Lematizando'):
        lemas.extend([t.lemma_.lower() for t in doc if t.is_alpha and len(t) > 2])
    return Counter([l for l in lemas if l not in stopwords]).most_common(n)


In [ ]:
# ============================================================
# CELL 13 — FRECUENCIAS SIN LEMATIZAR (equiv. cell 37)
# corpus + salud + cuentas_con_salud
# ============================================================

autores_con_salud = tweets_salud[COL_AUTOR].unique()
textos_cuentas_salud = corpus[corpus[COL_AUTOR].isin(autores_con_salud)]['Texto_limpio']

df_corpus_freq = pd.DataFrame(
    obtener_frecuencias(corpus['Texto_limpio'], STOP_WORDS, n=None),
    columns=['palabra', 'frecuencia_corpus']
)
df_salud_freq = pd.DataFrame(
    obtener_frecuencias(tweets_salud['Texto_limpio'], STOP_WORDS, n=None),
    columns=['palabra', 'frecuencia_salud']
)
df_cuentas_freq = pd.DataFrame(
    obtener_frecuencias(textos_cuentas_salud, STOP_WORDS, n=None),
    columns=['palabra', 'frecuencia_cuentas']
)

tabla_frecuencias = (
    df_corpus_freq
    .merge(df_salud_freq, on='palabra', how='outer')
    .merge(df_cuentas_freq, on='palabra', how='outer')
    .fillna(0)
)
tabla_frecuencias[['frecuencia_corpus','frecuencia_salud','frecuencia_cuentas']] = (
    tabla_frecuencias[['frecuencia_corpus','frecuencia_salud','frecuencia_cuentas']].astype(int)
)
tabla_frecuencias['proporcion_salud'] = tabla_frecuencias.apply(
    lambda r: r['frecuencia_salud'] / r['frecuencia_corpus'] if r['frecuencia_corpus'] > 0 else 0, axis=1
)
tabla_frecuencias['proporcion_cuentas'] = tabla_frecuencias.apply(
    lambda r: r['frecuencia_cuentas'] / r['frecuencia_corpus'] if r['frecuencia_corpus'] > 0 else 0, axis=1
)

tabla_top = tabla_frecuencias.sort_values('frecuencia_salud', ascending=False).head(30)
tabla_top.head(20)


## 10 · Tabla reducida para el artículo
Equivalente a la celda 38 del paper (Karen guarda LaTeX; aquí guardamos Excel).

In [ ]:
# ============================================================
# CELL 14 — TABLA REDUCIDA PARA EL ARTICULO (equiv. cell 38)
# Karen: LaTeX (.tex)  →  aqui: Excel (.xlsx)
# ============================================================

tabla_reducida = tabla_top[['palabra','frecuencia_corpus','frecuencia_cuentas','proporcion_cuentas']].copy()
tabla_reducida['proporcion_cuentas'] = tabla_reducida['proporcion_cuentas'].round(2)
tabla_reducida = tabla_reducida.rename(columns={
    'palabra'           : 'Palabra',
    'frecuencia_corpus' : 'Freq. corpus',
    'frecuencia_cuentas': 'Freq. cuentas con salud',
    'proporcion_cuentas': 'Proporcion cuentas',
})

ruta_reducida = DATA_PROCESSED / 'tabla_reducida_frecuencias.xlsx'
tabla_reducida.to_excel(ruta_reducida, index=False, engine='openpyxl')
print(f'[GUARDADO] tabla_reducida_frecuencias.xlsx')
tabla_reducida


## 11 · Frecuencias lematizadas *(opcional)*
Equivalente a las celdas 39–44 del paper.

> **Antes de correr:** elige una opción:
> - `LEMATIZAR = False` + `CARGAR = True` → carga desde parquet si ya existen
> - `LEMATIZAR = True`  + `CARGAR = False` → recalcula (puede tardar varios minutos)


In [ ]:
# ============================================================
# CELL 15 — FRECUENCIAS LEMATIZADAS (equiv. cells 41-42)
# ============================================================

LEMATIZAR = False   # True para recalcular
CARGAR    = True    # True para cargar desde parquet si existe

ruta_corpus_lem = DATA_PROCESSED / 'frecuencias_corpus_lematizado.parquet'
ruta_salud_lem  = DATA_PROCESSED / 'frecuencias_salud_lematizado.parquet'

if CARGAR and ruta_corpus_lem.exists() and ruta_salud_lem.exists():
    df_corpus_lem = pd.read_parquet(ruta_corpus_lem)
    df_salud_lem  = pd.read_parquet(ruta_salud_lem)
    print('Frecuencias lematizadas cargadas desde parquet')
elif LEMATIZAR:
    df_corpus_lem = pd.DataFrame(
        obtener_frecuencias(corpus['Texto_limpio'], STOP_WORDS, n=None, lematizar=True),
        columns=['palabra', 'frecuencia_corpus']
    )
    df_salud_lem = pd.DataFrame(
        obtener_frecuencias(tweets_salud['Texto_limpio'], STOP_WORDS, n=None, lematizar=True),
        columns=['palabra', 'frecuencia_salud']
    )
    print('Frecuencias lematizadas calculadas')
else:
    print('Skipped: establece LEMATIZAR=True o CARGAR=True')
    df_corpus_lem = df_corpus_freq.rename(columns={'frecuencia_corpus':'frecuencia_corpus'})
    df_salud_lem  = df_salud_freq.rename(columns={'frecuencia_salud':'frecuencia_salud'})


In [ ]:
# ============================================================
# CELL 16 — MERGE LEMATIZADO (equiv. cell 43)
# ============================================================

tabla_lem = (
    df_corpus_lem.merge(df_salud_lem, on='palabra', how='outer').fillna(0)
)
for col in ['frecuencia_corpus','frecuencia_salud']:
    if col in tabla_lem.columns:
        tabla_lem[col] = tabla_lem[col].astype(int)

if 'frecuencia_corpus' in tabla_lem and 'frecuencia_salud' in tabla_lem:
    tabla_lem['proporcion_salud'] = tabla_lem.apply(
        lambda r: r['frecuencia_salud'] / r['frecuencia_corpus'] if r['frecuencia_corpus'] > 0 else 0,
        axis=1
    )

tabla_lem_top = tabla_lem.sort_values('frecuencia_salud', ascending=False)
tabla_lem_top.head(10)


In [ ]:
# ============================================================
# CELL 17 — GUARDAR LEMATIZADOS EN PARQUET (equiv. cell 44)
# ============================================================

if LEMATIZAR:
    df_corpus_lem.to_parquet(ruta_corpus_lem, index=False)
    df_salud_lem.to_parquet(ruta_salud_lem, index=False)
    print('[GUARDADO] frecuencias_corpus_lematizado.parquet')
    print('[GUARDADO] frecuencias_salud_lematizado.parquet')
else:
    print('Skipped (LEMATIZAR=False)')


In [ ]:
# ============================================================
# CELL 18 — PALABRAS CON ALTA PROPORCION (equiv. cell 46)
# Karen: tabla_top[tabla_top['proporcion_columnas'] > 0.6]
# ============================================================

prop_base   = len(tweets_salud) / len(corpus)
umbral_prop = prop_base * 1.5

palabras_salud = (
    tabla_frecuencias[
        (tabla_frecuencias['proporcion_salud'] > umbral_prop) &
        (tabla_frecuencias['frecuencia_salud'] >= 10)
    ]
    .sort_values('frecuencia_salud', ascending=False)
    .head(30)
)

print(f'Proporcion base: {prop_base:.3f}  |  Umbral: {umbral_prop:.3f}')
print(f'Palabras con alta asociacion a salud: {len(palabras_salud)}')
palabras_salud


## 12 · Guardar todas las tablas

In [ ]:
# ============================================================
# CELL 19 — GUARDAR TABLAS FINALES
# ============================================================

ruta_excel = DATA_PROCESSED / 'tabla_frecuencias_salud.xlsx'

with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:
    tabla_subcats.to_excel(writer,             sheet_name='Subcategorias',         index=False)
    mensual.reset_index().to_excel(writer,     sheet_name='Temporal',              index=False)
    autores_merge.to_excel(writer,             sheet_name='Top_autores',           index=False)
    autores_por_subcat.reset_index().to_excel(writer, sheet_name='Autores_por_subcat', index=False)
    entidad_merge.to_excel(writer,             sheet_name='Por_tipo_cuenta',       index=False)
    tabla_top.to_excel(writer,                 sheet_name='Frecuencias_top30',     index=False)
    palabras_salud.to_excel(writer,            sheet_name='Palabras_salud',        index=False)
    df_ratio.to_excel(writer,                  sheet_name='Proporcion_por_autor',  index=False)

print(f'[GUARDADO] tabla_frecuencias_salud.xlsx  ({ruta_excel})')
print()
print('Notebook 05 completado.')
print('Siguiente -> 06_NER.ipynb')
